# Preprocessing & EDA
## Healthcare Sector Focus

**Goal**: Analyse the UK ICO data security incident trends with a focus on the healthcare sector to prepare for building ML models and a dashboard.

**Issues to address**: 
- Multiple rows assigned to one power BI reference (same incident, multiple characteristics).
- Definition change in april 2021 for "informal action" and "no further action".
- Unknown/ unassigned values.
- Categorical data needs encoding

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# load the dataset
df_original = pd.read_csv("data-security-incidents-trends-q1-2019-to-q2-2025.csv", delimiter = ",")

# inital look at data
df_original.info()

In [ ]:
# count all sectors
sector_counts = df_original['Sector'].value_counts()
sector_perc = sector_counts / len(df_original) * 100

plt.figure(figsize=(10, 8))
bars = plt.bar(sector_counts.index, sector_counts.values)

plt.title("Sector Distribution (with Percentages)")
plt.xlabel("Sector")
plt.ylabel("Row Count")
plt.xticks(rotation=45, ha='right')

# add percentage labels on top of each bar
for bar, pct in zip(bars, sector_perc):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2, # put percentage in the middle
        height, f"{pct:.1f}%",
        ha='center', va='bottom'
    )

print(f"\nTotal sectors: {df_original['Sector'].nunique()}")

plt.tight_layout()
plt.show()

In [ ]:
# filtering the database to focus of the health sector
df_health = df_original.copy(deep=True)
df_health = df_original[df_original["Sector"] == "Health"] 

# standardising the column names
df_health.columns = df_health.columns.str.strip().str.lower().str.replace(" ", "_").str.replace(".", "")

# checking how many missing values are in the healthcare df
df_health.isin(["Unknown", "unknown", "UNKNOWN"]).sum()

# look at new healthcare sector focused df
df_health.info()

In [ ]:
# finding the unique vs total BI references 
total_rows = len(df_health)-1 # -1 for column header
unique_bi_refs = df_health['bi_reference'].nunique()
duplicate_ratio = (total_rows - unique_bi_refs) / total_rows * 100

print(f"\nTotal rows in the dataset: {total_rows:,}")
print(f"Unique BI References: {unique_bi_refs:,}")
print(f"Rows representing duplicate characteristics: {total_rows - unique_bi_refs:,}")
print(f"Duplication rate: {duplicate_ratio:.2f}%")

In [ ]:
# plot: pie chart of unique vs duplicate references
bi_ref_counts = df_health['bi_reference'].value_counts()
single_row = (bi_ref_counts == 1).sum()
multiple_rows = (bi_ref_counts > 1).sum()
plt.pie([single_row, multiple_rows], 
            labels=['Single Row\n(No duplication)', 'Multiple Rows\n(Duplicated)'], 
            autopct='%1.1f%%', colors=['#ff9999', '#66b3ff'], startangle=90,
            textprops={'fontsize': 11, 'fontweight': 'bold'})
plt.title('BI References (Health Sector): Single vs Multiple Rows', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# analyse which columns typically vary across duplicates
duplicated_bi_refs= df_health[df_health['bi_reference'].duplicated(keep=False)]

# nunique per group per column
varying_columns = duplicated_bi_refs.groupby("bi_reference").nunique()

# a column varies in a group if nunique > 1
varying_counts = (varying_columns > 1).sum()

total_groups = duplicated_bi_refs['bi_reference'].nunique()
result = (varying_counts
    .sort_values(ascending=False)
    .rename("count")
    .to_frame()
)

result["percentage"] = (result["count"] / total_groups * 100)
print(f"Columns that vary across duplicate rows:\n {result}")